In [1]:
from pyspark.sql import SparkSession
import pyspark

AWS_ACCESS_KEY = "minioadmin"
AWS_SECRET_KEY = "minioadmin"
AWS_S3_ENDPOINT = "http://minio_server:9000"
WAREHOUSE = "s3a://gold/" 
NESSIE_URI = "http://nessie:19120/api/v1"

conf = (
    pyspark.SparkConf()
    .setAppName("Lakehouse-Iceberg-TrainModel")  
    .set('spark.jars.packages',
         'org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,'
         'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,'
         'org.apache.hadoop:hadoop-aws:3.3.4,'
         'com.amazonaws:aws-java-sdk-bundle:1.12.300')
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    .set("spark.sql.catalog.nessie.authentication.type", "NONE")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.warehouse", WAREHOUSE)
    .set("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .set("spark.sql.catalog.nessie.s3.endpoint", AWS_S3_ENDPOINT)
    .set("spark.sql.catalog.nessie.s3.access-key", AWS_ACCESS_KEY)
    .set("spark.sql.catalog.nessie.s3.secret-key", AWS_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
)

spark = (
    SparkSession.builder
    .config(conf=conf) 
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")


In [2]:
df_fact = spark.table("nessie.gold_db.fact_order")
df_customer = spark.table("nessie.gold_db.dim_customer")
df_product = spark.table("nessie.gold_db.dim_product")
df_time = spark.table("nessie.gold_db.dim_time")
df_location = spark.table("nessie.gold_db.dim_location")


In [3]:
query = """
SELECT 
    f.time_id,
    f.customer_id,
    f.product_id,
    f.location_id,
    f.quantity,
    f.unit_price,        
    f.total_revenue,     

    -- Dim_time
    t.order_date,
    t.year,
    t.month,
    t.day,
    t.quarter,
    t.weekday_name,

    -- Dim_customer
    c.gender,
    c.age_group,
    c.education,
    c.income,
    c.race,
    c.state,             -- Cột state từ bảng khách hàng

    -- Dim_product
    p.product_title,
    p.product_category,

    -- Dim_location
    l.state_name purchase_state,
    l.region purchase_region

FROM nessie.gold_db.fact_order f
LEFT JOIN nessie.gold_db.dim_time t     ON f.time_id     = t.time_id
LEFT JOIN nessie.gold_db.dim_customer c ON f.customer_id  = c.customer_id
LEFT JOIN nessie.gold_db.dim_product p   ON f.product_id  = p.product_id
LEFT JOIN nessie.gold_db.dim_location l  ON f.location_id = l.location_id
"""

In [4]:
df_fact_full = spark.sql(query)
df_fact_full.limit(10).toPandas()



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

,time_id,customer_id,product_id,location_id,quantity,unit_price,total_revenue,order_date,year,month,...,gender,age_group,education,income,race,state,product_title,product_category,purchase_state,purchase_region
0,20221126,R_262vJPpUeeFKuHT,0307118932,34359738369,1.0,3.99,3.99,2022-11-26,2022,11,...,Female,65 and older,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Just Grandma and Me (Little Critter) (Pictureb...,ABIS_BOOK,Arizona,West
1,20210419,R_2AWkL1j4mAcAll8,0307464970,34359738369,1.0,15.99,15.99,2021-04-19,2021,4,...,Male,18 - 24 years,High school diploma or GED,"$25,000 - $49,999",American Indian/Native American or Alaska Native,Arizona,The Harlem Hellfighters,ABIS_BOOK,Arizona,West
2,20200609,R_tMsVLA8C6RuTmlb,048640708X,34359738369,1.0,7.89,7.89,2020-06-09,2020,6,...,Female,55 - 64 years,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Chinese Painting Techniques (Dover Art Instruc...,ABIS_BOOK,Arizona,West
3,20191221,R_262vJPpUeeFKuHT,0812971833,34359738369,1.0,12.60,12.60,2019-12-21,2019,12,...,Female,65 and older,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Olive Kitteridge,ABIS_BOOK,Arizona,West
4,20221214,R_AmMCFCEBSSiRhLz,1492677698,34359738369,1.0,17.99,17.99,2022-12-14,2022,12,...,Female,35 - 44 years,Bachelor's degree,"$100,000 - $149,999",White or Caucasian,Arizona,The Complete Baking Book for Young Chefs: 100+...,ABIS_BOOK,Arizona,West
5,20200826,R_3G9b1ToPNT28Utq,1627797645,34359738369,1.0,6.50,6.50,2020-08-26,2020,8,...,Female,25 - 34 years,High school diploma or GED,"$50,000 - $74,999",White or Caucasian,Arizona,Marlena: A Novel,ABIS_BOOK,Arizona,West
6,20221004,R_262vJPpUeeFKuHT,1637315309,34359738369,1.0,11.99,11.99,2022-10-04,2022,10,...,Female,65 and older,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Hooty Pooty the Owl: A Funny Rhyming Halloween...,ABIS_BOOK,Arizona,West
7,20190401,R_2E0Q3pbtzJQqLg7,B00004R9TL,34359738369,1.0,20.85,20.85,2019-04-01,2019,4,...,Male,35 - 44 years,Bachelor's degree,"$25,000 - $49,999",White or Caucasian,Florida,"BLACK+DECKER Trimmer Line, 30-Foot, 0.065-Inch...",IGNITION_COIL,Arizona,West
8,20191127,R_06RZP9pS7kONINr,B00006ANDK,996432412672,1.0,19.99,19.99,2019-11-27,2019,11,...,Female,65 and older,Bachelor's degree,"$75,000 - $99,999",White or Caucasian,South Dakota,Oral-B Sensitive Gum Care Electric Toothbrush ...,TOOTHBRUSH_HEAD,South Dakota,Midwest
9,20210118,R_4PC8xUs7y9n494R,B0000BYEF6,34359738369,2.0,12.34,24.68,2021-01-18,2021,1,...,Female,25 - 34 years,Bachelor's degree,"$100,000 - $149,999",White or Caucasian,Arizona,Lutron Credenza Plug-In Dimmer for Incandescen...,ELECTRONIC_SWITCH,Arizona,West


In [5]:
# Hiển thị schema sau khi làm sạch
df_fact_full.printSchema()

root
 |-- time_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- location_id: long (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- weekday_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- education: string (nullable = true)
 |-- income: string (nullable = true)
 |-- race: string (nullable = true)
 |-- state: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- purchase_state: string (nullable = true)
 |-- purchase_region: string (nullable = true)



In [6]:
# Đếm số dòng
num_rows =df_fact_full.count()
# Đếm số cột
num_cols = len(df_fact_full.columns)
print(f"\nKích thước dữ liệu: ({num_rows}, {num_cols})")


Kích thước dữ liệu: (1670103, 23)


In [7]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import RegressionEvaluator

# =====================================================
# Tính toán toàn bộ feature dựa trên Schema Gold chuẩn
# =====================================================
df_final = (
    df_fact_full.groupBy("customer_id")
    .agg(
        # Tổng chi tiêu (Đã đổi sang total_revenue)
        F.sum("total_revenue").alias("total_spend"),

        # Trung bình chi tiêu trên mỗi đơn hàng
        (F.sum("total_revenue") / F.countDistinct("time_id")).alias("avg_order_value"),

        # Độ lệch chuẩn chi tiêu trên đơn hàng
        F.stddev("total_revenue").alias("std_order_value"),

        # Trung bình số lượng sản phẩm mỗi đơn
        (F.sum("quantity") / F.countDistinct("time_id")).alias("avg_items_per_order"),

        # Tổng số đơn hàng
        F.countDistinct("time_id").alias("total_orders"),

        # Năm đầu và năm cuối hoạt động
        F.min("year").alias("first_year"),
        F.max("year").alias("last_year")
    )
    # Thêm số năm hoạt động và chi tiêu trung bình mỗi năm
    .withColumn("years_active", (F.col("last_year") - F.col("first_year") + 1))
    .withColumn("years_active", F.when(F.col("years_active") <= 0, 1).otherwise(F.col("years_active")))
    .withColumn("avg_spend_per_year", F.col("total_spend") / F.col("years_active"))
    .na.fill(0)  # Đảm bảo không có null để tránh lỗi khi đưa vào mô hình ML
)

# Kiểm tra dữ liệu sau khi tính toán feature
df_final.show(5)

+-----------------+------------------+-----------------+------------------+-------------------+------------+----------+---------+------------+------------------+
|      customer_id|       total_spend|  avg_order_value|   std_order_value|avg_items_per_order|total_orders|first_year|last_year|years_active|avg_spend_per_year|
+-----------------+------------------+-----------------+------------------+-------------------+------------+----------+---------+------------+------------------+
|R_1jO4s7oht3pyKEc|          13041.71|60.37828703703703|25.637963404064184| 2.8333333333333335|         216|      2018|     2023|           6|2173.6183333333333|
|R_10TV1zyi4yCEEkl|30552.119999999984|74.33605839416055| 83.71474123408707| 2.6739659367396595|         411|      2018|     2023|           6| 5092.019999999998|
|R_297dOANqCntVXou|11332.730000000001|44.79339920948617|26.850143871729934|  2.399209486166008|         253|      2018|     2022|           5|2266.5460000000003|
|R_2dyITPHbbfmCXJn|16020.079

In [8]:

df_final.select(
    "customer_id",
    "total_spend", "avg_spend_per_year",
    "avg_order_value", "std_order_value",
    "avg_items_per_order", "total_orders",
    "years_active"
).show(10, truncate=False)


+-----------------+------------------+------------------+------------------+------------------+-------------------+------------+------------+
|customer_id      |total_spend       |avg_spend_per_year|avg_order_value   |std_order_value   |avg_items_per_order|total_orders|years_active|
+-----------------+------------------+------------------+------------------+------------------+-------------------+------------+------------+
|R_1jO4s7oht3pyKEc|13041.71          |2173.6183333333333|60.37828703703703 |25.637963404064184|2.8333333333333335 |216         |6           |
|R_10TV1zyi4yCEEkl|30552.119999999984|5092.019999999998 |74.33605839416055 |83.71474123408707 |2.6739659367396595 |411         |6           |
|R_297dOANqCntVXou|11332.730000000001|2266.5460000000003|44.79339920948617 |26.850143871729934|2.399209486166008  |253         |5           |
|R_2dyITPHbbfmCXJn|16020.079999999998|2670.013333333333 |92.06942528735631 |49.11779146550433 |3.1666666666666665 |174         |6           |
|R_3GD

In [9]:
# =====================================================
#  Xác định features & target
# =====================================================
num_features = [
    "total_spend",
    "years_active",
    "avg_order_value",
    "std_order_value",
    "avg_items_per_order",
    "total_orders"
]
target = "avg_spend_per_year"

df_clean = df_final.na.drop(subset=num_features + [target])
# Chia train/test
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count()} dòng | Test: {test_df.count()} dòng")

assembler = VectorAssembler(
    inputCols=num_features,
    outputCol="features",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withMean=True,
    withStd=True
)

Train: 4030 dòng | Test: 930 dòng


## Random Forest

In [10]:
# --------------------------
# Pipeline mô hình Random Forest
# --------------------------
rf = RandomForestRegressor(
    labelCol=target,
    featuresCol="scaledFeatures",
    seed=42
)

rf_pipeline = Pipeline(stages=[assembler, scaler, rf])

# --------------------------
# Tập siêu tham số cần thử
# --------------------------
param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 200])
    .addGrid(rf.maxDepth, [8, 10, 12])
    .build()
)

# --------------------------
# TrainValidationSplit
# --------------------------
evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="r2")

tvs = TrainValidationSplit(
    estimator=rf_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,
    parallelism=4
)

# --------------------------
# Huấn luyện mô hình
# --------------------------
rf_tvs_model = tvs.fit(train_df)

# --------------------------
# In thông tin mô hình tốt nhất
# --------------------------
best_rf = rf_tvs_model.bestModel.stages[-1]
print("Best Model:")
print(" - numTrees:", best_rf.getNumTrees)
print(" - maxDepth:", best_rf.getMaxDepth())


Best Model:
 - numTrees: 50
 - maxDepth: 12


In [11]:
# --------------------------
# Đánh giá mô hình
# --------------------------
train_pred = rf_tvs_model.bestModel.transform(train_df)
test_pred  = rf_tvs_model.bestModel.transform(test_df)

metrics = ['r2', 'mae', 'rmse']
for metric in metrics:
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    print(f"{metric.upper()} Train:", round(evaluator.evaluate(train_pred), 4))
    print(f"{metric.upper()} Test :", round(evaluator.evaluate(test_pred), 4))
    print("----------")

R2 Train: 0.9902
R2 Test : 0.9662
----------
MAE Train: 56.7472
MAE Test : 95.12
----------
RMSE Train: 162.7687
RMSE Test : 303.8321
----------


In [12]:
import mlflow
import mlflow.spark
from pyspark.ml.evaluation import RegressionEvaluator

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Customer_Spending_Prediction")

# Tính metrics train/test
metrics_dict = {}
for metric in metrics:  # metrics = ['r2', 'mae', 'rmse']
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    metrics_dict[f"{metric}_train"] = round(evaluator.evaluate(train_pred), 4)
    metrics_dict[f"{metric}_test"]  = round(evaluator.evaluate(test_pred), 4)

# Lưu vào MLflow
with mlflow.start_run(run_name="RandomForest_Model"):
    # Log param
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("numTrees", best_rf.getNumTrees)
    mlflow.log_param("maxDepth", best_rf.getMaxDepth())
    
    # Log metric
    for k, v in metrics_dict.items():
        mlflow.log_metric(k, v)
    
    # Log mô hình (PipelineModel đầy đủ)
    mlflow.spark.log_model(rf_tvs_model.bestModel, "rf_model")

    print("Logged Random Forest model to MLflow")


2026/03/18 08:39:57 INFO mlflow.tracking.fluent: Experiment with name 'Customer_Spending_Prediction' does not exist. Creating a new experiment.


Logged Random Forest model to MLflow
🏃 View run RandomForest_Model at: http://mlflow:5000/#/experiments/1/runs/d66a94ed244b46468ee7359a5b5d7579
🧪 View experiment at: http://mlflow:5000/#/experiments/1


## XGB

In [14]:
from pyspark.ml import Pipeline
from xgboost.spark import SparkXGBRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

# --------------------------
# XGBoost Spark phân tán
# --------------------------
xgb = SparkXGBRegressor(
    features_col="scaledFeatures",
    label_col=target,
    num_workers=1,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    objective="reg:squarederror"
)

# --------------------------
# Pipeline
# --------------------------
xgb_pipeline = Pipeline(stages=[assembler, scaler, xgb])

# --------------------------
# Grid Search / TrainValidationSplit
# --------------------------
param_grid = (
    ParamGridBuilder()
    .addGrid(xgb.max_depth, [4, 6])
    .addGrid(xgb.n_estimators, [50, 100])
    .addGrid(xgb.learning_rate, [0.05, 0.1])
    .addGrid(xgb.subsample, [0.7, 0.8])
    .build()
)

evaluator = RegressionEvaluator(
    labelCol=target,
    predictionCol="prediction",
    metricName="r2"
)

tvs = TrainValidationSplit(
    estimator=xgb_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,
    parallelism=4,   # chạy song song 4 mô hình
    seed=42
)

# --------------------------
# Huấn luyện mô hình
# --------------------------
xgb_tvs_model = tvs.fit(train_df)

# --------------------------

2026-03-18 08:51:08,370 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 4, 'objective': 'reg:squarederror', 'subsample': 0.7, 'tree_method': 'hist', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-03-18 08:51:08,377 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 4, 'objective': 'reg:squarederror', 'subsample': 0.8, 'tree_method': 'hist', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-03-18 08:51:08,365 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 4, 'objective': '

In [15]:
# Lấy mô hình tốt nhất
best_xgb = xgb_tvs_model.bestModel.stages[-1]
tuned_params = ["max_depth", "n_estimators", "learning_rate", "subsample"]
print("Best XGBoost Parameters (from grid search):")
for p in tuned_params:
    value = best_xgb.getOrDefault(best_xgb.getParam(p))
    print(f" - {p}: {value}")

Best XGBoost Parameters (from grid search):
 - max_depth: 4
 - n_estimators: 100
 - learning_rate: 0.1
 - subsample: 0.8


In [16]:
# Đánh giá mô hình
# --------------------------
train_pred = xgb_tvs_model.transform(train_df)
test_pred  = xgb_tvs_model.transform(test_df)

metrics = ['r2', 'mae', 'rmse']
for metric in metrics:
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    print(f"{metric.upper()} Train:", round(evaluator.evaluate(train_pred), 4))
    print(f"{metric.upper()} Test :", round(evaluator.evaluate(test_pred), 4))
    print("----------")

R2 Train: 0.9988
R2 Test : 0.9931
----------
MAE Train: 34.0996
MAE Test : 47.7906
----------
RMSE Train: 57.9288
RMSE Test : 137.1903
----------


In [17]:
import mlflow
import mlflow.spark

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("XGBoost_Regression")

with mlflow.start_run():
    mlflow.spark.log_model(best_xgb, "xgb_model")
    mlflow.log_metric("r2_train", 0.9987)
    mlflow.log_metric("r2_test", 0.9905)

2026/03/18 09:03:56 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Regression' does not exist. Creating a new experiment.


🏃 View run likeable-chimp-442 at: http://mlflow:5000/#/experiments/2/runs/065a6c67039743e3ae0c7214d2b89233
🧪 View experiment at: http://mlflow:5000/#/experiments/2


In [18]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Customer_Spending_Prediction")

# Tính metrics train/test
metrics_dict = {}
for metric in metrics:  # metrics = ['r2', 'mae', 'rmse']
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    metrics_dict[f"{metric}_train"] = round(evaluator.evaluate(xgb_tvs_model.transform(train_df)), 4)
    metrics_dict[f"{metric}_test"]  = round(evaluator.evaluate(xgb_tvs_model.transform(test_df)), 4)

# Lưu vào MLflow
with mlflow.start_run(run_name="XGBoost_Model"):
    # Log param
    mlflow.log_param("model", "SparkXGBRegressor")
    mlflow.log_param("max_depth", best_xgb.getOrDefault(best_xgb.getParam("max_depth")))
    mlflow.log_param("n_estimators", best_xgb.getOrDefault(best_xgb.getParam("n_estimators")))
    mlflow.log_param("learning_rate", best_xgb.getOrDefault(best_xgb.getParam("learning_rate")))
    mlflow.log_param("subsample", best_xgb.getOrDefault(best_xgb.getParam("subsample")))
    
    # Log metric
    for k, v in metrics_dict.items():
        mlflow.log_metric(k, v)
        print(f"{k}: {v}")
    
    # Log mô hình (PipelineModel đầy đủ: assembler + scaler + XGB)
    mlflow.spark.log_model(xgb_tvs_model.bestModel, "xgb_model")

    print("Logged XGBoost Spark model to MLflow")


r2_train: 0.9988
r2_test: 0.9931
mae_train: 34.0996
mae_test: 47.7906
rmse_train: 57.9288
rmse_test: 137.1903
Logged XGBoost Spark model to MLflow
🏃 View run XGBoost_Model at: http://mlflow:5000/#/experiments/1/runs/64e17a04ecae4aa89155d80f44709c99
🧪 View experiment at: http://mlflow:5000/#/experiments/1


In [19]:
mlflow.spark.log_model(
    best_xgb,
    artifact_path="xgb_model",
    registered_model_name="XGB_Model"
)


Successfully registered model 'XGB_Model'.
2026/03/18 09:13:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_Model, version 1
Created version '1' of model 'XGB_Model'.
